# Import libraries

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer

# Load Dataset

In [2]:
data = load_breast_cancer()

In [10]:
x = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

In [11]:
x.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [12]:
y.head()

0    0
1    0
2    0
3    0
4    0
dtype: int32

In [9]:
# so this data that we import from sklearn doesnt need to clean or feature extracting it is already clean and well
#   orgainised data

# Train the model and split the data

In [13]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [25]:
models = {'RandomForest': {'model': RandomForestClassifier(),
                            'params': {'n_estimators': [50, 100, 150],
                                        'max_depth': [None, 10, 20],
                                        'min_samples_split': [2, 5, 10]}},
            'SVM': {'model': SVC(),
                    'params': {'C': [0.1, 1, 10],
                                'kernel': ['linear', 'rbf'],
                                'gamma': ['scale', 'auto']}},
            'LogisticRegression': {'model': LogisticRegression(max_iter=1000),
                                    'params': {'C': [0.1, 1, 10],
                                    'solver': ['liblinear', 'lbfgs']}}}

# gridsearch and result display

In [27]:
results = []

for name, mp in models.items():
    clf = GridSearchCV(mp['model'], mp['params'], cv=5, scoring='f1', n_jobs=-1)
    clf.fit(x_train, y_train)
    y_pred = clf.predict(x_test)
    
    results.append({
        'Model': name,
        'Best Params': clf.best_params_,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred)})

In [28]:
pd.DataFrame(results)

,Model,Best Params,Accuracy,Precision,Recall,F1 Score
0,RandomForest,"{'max_depth': None, 'min_samples_split': 2, 'n...",0.964912,0.958904,0.985915,0.972222
1,SVM,"{'C': 1, 'gamma': 'scale', 'kernel': 'linear'}",0.956140,0.945946,0.985915,0.965517
2,LogisticRegression,"{'C': 10, 'solver': 'liblinear'}",0.956140,0.945946,0.985915,0.965517


In [29]:
# for faster tuning

In [32]:
from sklearn.model_selection import RandomizedSearchCV

In [33]:
results = []

for name, mp in models.items():
    clf = RandomizedSearchCV(mp['model'], mp['params'], cv=5, scoring='f1', n_iter=10, n_jobs=-1, random_state=42)
    clf.fit(x_train, y_train)
    y_pred = clf.predict(x_test)
    
    results.append({
        'Model': name,
        'Best Params': clf.best_params_,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred)})

C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 6 is smaller than n_iter=10. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


In [34]:
pd.DataFrame(results)

,Model,Best Params,Accuracy,Precision,Recall,F1 Score
0,RandomForest,"{'n_estimators': 50, 'min_samples_split': 5, '...",0.964912,0.958904,0.985915,0.972222
1,SVM,"{'kernel': 'linear', 'gamma': 'scale', 'C': 1}",0.956140,0.945946,0.985915,0.965517
2,LogisticRegression,"{'solver': 'liblinear', 'C': 10}",0.956140,0.945946,0.985915,0.965517


# at last best model selection

In [37]:
result_df = pd.DataFrame(results)
best_model = result_df.loc[result_df['F1 Score'].idxmax()]
print("Best Model:\n", best_model)

Best Model:
 Model                                               RandomForest
Best Params    {'n_estimators': 50, 'min_samples_split': 5, '...
Accuracy                                                0.964912
Precision                                               0.958904
Recall                                                  0.985915
F1 Score                                                0.972222
Name: 0, dtype: object


In [38]:
# therefor our best model for this dataset is random forest